# Spike probability prediction using Cascade (Rupprecht et al. 2021)

This notebook runs with the cascade environment. It should be run after aggregating the different experiments with the IHC-GlobalAnalysis.ipynb notebook

In [4]:
CASCADE_PATH = '../../../CascadeTorch'

In [6]:
%pylab
%load_ext autoreload
%autoreload 2
%matplotlib inline
import pandas as pd
import os
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
from scipy.signal import argrelextrema,argrelmin,argrelmax
import sys
sys.path.append('../../src')
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown
from traceUtilities import fillMissingValues, stackedPlot, rollingMedianCorrection
import ast 
from tifffile import imread
#Load cascade
import sys
sys.path.append(CASCADE_PATH)



from cascade2p import checks
checks.check_packages()
from cascade2p import cascade # local folder
from cascade2p.utils import plot_dFF_traces, plot_noise_level_distribution, plot_noise_matched_ground_truth
import ruamel.yaml as yaml
yaml = yaml.YAML(typ='rt')



parameterFolder = '../../parameters/'
fileHeader = [
'alpha9'

]

alldatalist  = []
for h in fileHeader:
    inputFilename = os.path.join('../../',h)+'.xlsx'

    alldata1 = pd.read_excel(inputFilename)
    alldatalist.append(alldata1[alldata1['discard']!=1])

alldata = pd.concat(alldatalist, ignore_index=True)

alldata = alldata[~alldata['Folder'].isna()]
alldata = alldata.reset_index()
alldata['Mouse ID'] = alldata['Strain'] + alldata['Mouse ID'].astype(str)

alldata = alldata.sort_values(['Age','Mouse ID','Folder']).reset_index()


resultsFOlder = '../../analysis_results/'

#Chose the appropriate fileheader. 'Myo15+Atoh_IHCs_events' is the output of IHC-GlobalAnalysis.ipynb, while IHC-GlobalAnalysis_processed is after further analysis with IHC-GlobalAnalysis-part2
fileHeader = 'alpha9'

inputFilename = os.path.join(resultsFOlder,fileHeader)+'.xlsx'

tempFilename = '../../analysis_results/master_temp.csv'

#Change drive 
localDrive = 'E'
alldata['Folder'] = localDrive + alldata['Folder'].str[1:]



Using matplotlib backend: inline
%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
	YAML reader installed (version 0.19.1).
	Torch installed (version 2.10.0+cu128).


/home/marcotti-lab/git-repos/In-Vivo-Analysis-core/.venv/lib/python3.12/site-packages/IPython/core/magics/pylab.py:166: UserWarning: pylab import has clobbered these variables: ['imread']
`%matplotlib` prevents importing * from pylab and numpy
  warn("pylab import has clobbered these variables: %s"  % clobbered +


In [9]:
#Load previous result

master = pd.read_excel(inputFilename)

master['Peak positions'] = master['Peak positions'].apply(ast.literal_eval)

alltraces = pd.read_parquet(os.path.join(resultsFOlder,f'{fileHeader}_events_alltraces.parquet'))
master = master[master['Cell type']==1]


## Cascade


In [10]:
cascade.download_model?

Signature:
cascade.download_model(
    model_name,
    model_folder='Pretrained_models',
    info_file_link='https://raw.githubusercontent.com/PTRRupprecht/CascadeTorch/refs/heads/master/Pretrained_models/available_models_CascadeTorch.yaml',
    verbose=1,
)
Docstring:
Download and unzip pretrained model from the online repository

Parameters
----------
model_name : str
    Name of the model, e.g. 'Global_EXC_30Hz_smoothing25ms'
    This name has to correspond to a pretrained model that is available for download
    To see available models, run this function with model_name='update_models' and
    check the downloaded file 'available_models_CascadeTorch.yaml'

model_folder: str
    Absolute or relative path, which defines the location of the specified model_name folder
    Default value 'Pretrained_models' assumes a current working directory in the Cascade folder

info_file_link: str
    Direct download link to yaml file which contains download links for new models.
    Default value i

In [13]:

cascade.download_model( 'update_models',os.path.join(CASCADE_PATH,'Pretrained_models'),verbose = 1)

yaml_file = open(os.path.join(CASCADE_PATH,'Pretrained_models','available_models_CascadeTorch.yaml'))
X = yaml.load(yaml_file)
list_of_models = list(X.keys())
print('\n List of available models: \n')
for model in list_of_models:
  print(model)

You can now check the updated available_models_CascadeTorch.yaml file for valid model names.
File location: /home/marcotti-lab/git-repos/CascadeTorch/Pretrained_models/available_models_CascadeTorch.yaml

 List of available models: 

GC8f_EXC_100Hz_smoothing10ms
GC8f_EXC_15Hz_smoothing100ms_high_noise
GC8f_EXC_15Hz_smoothing50ms_high_noise
GC8f_EXC_30Hz_smoothing25ms_high_noise
GC8f_EXC_30Hz_smoothing50ms_high_noise
GC8f_EXC_7.5Hz_smoothing100ms_high_noise
GC8f_EXC_7.5Hz_smoothing200ms_high_noise
GC8m_EXC_15Hz_smoothing100ms_high_noise
GC8m_EXC_15Hz_smoothing50ms_high_noise
GC8m_EXC_30Hz_smoothing25ms_high_noise
GC8m_EXC_30Hz_smoothing50ms_high_noise
GC8m_EXC_7.5Hz_smoothing100ms_high_noise
GC8m_EXC_7.5Hz_smoothing200ms_high_noise
GC8s_EXC_15Hz_smoothing100ms_high_noise
GC8s_EXC_15Hz_smoothing50ms_high_noise
GC8s_EXC_20Hz_smoothing30ms_high_noise
GC8s_EXC_20Hz_smoothing60ms_high_noise
GC8s_EXC_30Hz_smoothing25ms_high_noise
GC8s_EXC_30Hz_smoothing50ms_high_noise
GC8s_EXC_3Hz_smoothing250

In [ ]:
#Associate each fps with a model

models = ['Global_EXC_25Hz_smoothing100ms', 'Global_EXC_30Hz_smoothing100ms','Global_EXC_40Hz_smoothing50ms','Global_EXC_20Hz_smoothing100ms','Global_EXC_30Hz_smoothing25ms']
for model_name in models:
#    model_name = "Global_EXC_30Hz_smoothing25ms_causalkernel" #@param {type:"string"}
    model_folder = os.path.join(CASCADE_PATH,'Pretrained_models')
    cascade.download_model( model_name,verbose = 1,model_folder=model_folder)

Pretrained model was saved in folder "/home/marcotti-lab/git-repos/CascadeTorch/Pretrained_models/Global_EXC_25Hz_smoothing100ms"
Pretrained model was saved in folder "/home/marcotti-lab/git-repos/CascadeTorch/Pretrained_models/Global_EXC_30Hz_smoothing100ms"
Pretrained model was saved in folder "/home/marcotti-lab/git-repos/CascadeTorch/Pretrained_models/Global_EXC_40Hz_smoothing50ms"
Pretrained model was saved in folder "/home/marcotti-lab/git-repos/CascadeTorch/Pretrained_models/Global_EXC_20Hz_smoothing200ms_causalkernel"
Pretrained model was saved in folder "/home/marcotti-lab/git-repos/CascadeTorch/Pretrained_models/Global_EXC_30Hz_smoothing25ms"


In [21]:
allSpikesByFolder = pd.DataFrame()
for folder in master['Folder'].unique():
    el = master[master['Folder']==folder]
    ids = el['Cell ID']
    traces = alltraces[ids].dropna().values.T
    traces = rollingMedianCorrection(traces.T,1000).T
    print('Number of neurons in dataset:', traces.shape[0])
    print('Number of timepoints in dataset:', traces.shape[1])
    if (el['fps'].values[0]==24.48) or (el['fps'].values[0]==27.163):
        model_name = 'Global_EXC_25Hz_smoothing100ms'
    elif (el['fps'].values[0]==22.279) or (el['fps'].values[0]==22.28):
         model_name = 'Global_EXC_20Hz_smoothing100ms'
    elif (el['fps'].values[0]==30.506) or (el['fps'].values[0]==34.787) or (el['fps'].values[0]==94.422):
        model_name = 'Global_EXC_30Hz_smoothing100ms'
    elif el['fps'].values[0]==40.467:
         model_name = 'Global_EXC_40Hz_smoothing50ms'
    else:
        print('No right model')
        break
    if  (el['fps'].values[0]==94.422):
        traces = traces[:,::3]
    #select pretrained model
    model_name = 'Global_EXC_30Hz_smoothing100ms' # Only one available at the moment
    spike_prob = cascade.predict(model_name,traces,model_folder=model_folder)
    allSpikesByFolder[ids] = pad(spike_prob.T,((0,30000-spike_prob.shape[1]),(0,0)),mode='constant',constant_values = np.nan)


Number of neurons in dataset: 17
Number of timepoints in dataset: 11808

 
The selected model was trained on 18 datasets, with 5 ensembles for each noise level, at a sampling rate of 30Hz, with a resampled ground truth that was smoothed with a Gaussian kernel of a standard deviation of 200 milliseconds. 
 

Loaded model was trained at frame rate 30 Hz
Given argument traces contains 17 neurons and 11808 frames.
Noise levels (mean, std; in standard units): 0.19, 0.02

Predictions for noise level 1:
	... ensemble 0
	... ensemble 1
	... ensemble 2
	... ensemble 3
	... ensemble 4

Predictions for noise level 2:
	No neurons for this noise level

Predictions for noise level 3:
	No neurons for this noise level

Predictions for noise level 4:
	No neurons for this noise level

Predictions for noise level 5:
	No neurons for this noise level

Predictions for noise level 6:
	No neurons for this noise level

Predictions for noise level 7:
	No neurons for this noise level

Predictions for noise level

KeyboardInterrupt: 

In [6]:
#Uncomment if you want to save
allSpikesByFolder.to_csv('..\\..\\analysis_results\\Myo15_IHCs_ex_vivo_events_alltraces_cascadespikeprob_rollMedCorr1000.csv')